# 📚 賢者とユイの読書倶楽部 — eBook自動生成

**使い方（3ステップ）：**
1. 「ランタイム」→「すべてのセルを実行」をクリック
2. Gemini APIキーを貼り付けて Enter
3. 本のタイトルを入力して Enter

→ Markdown・EPUB・Word の3ファイルが自動生成されます

In [ ]:
# ── ステップ0：ライブラリインストール ──────────────────────────────
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'google-generativeai', 'ebooklib', 'Pillow', 'python-docx'],
    check=True
)
print('✅ インストール完了')

In [ ]:
# ── ステップ1：Gemini API キーを入力 ──────────────────────────────
import getpass, google.generativeai as genai

GEMINI_API_KEY = getpass.getpass('Gemini APIキーを入力してください: ')
genai.configure(api_key=GEMINI_API_KEY)

# 利用できるモデルを自動検出
print('🔍 利用可能なモデルを確認中...')
PREFERRED = ['gemini-2.0-flash', 'gemini-1.5-flash', 'gemini-1.5-pro', 'gemini-pro']
available = {m.name.split('/')[-1] for m in genai.list_models()
             if 'generateContent' in m.supported_generation_methods}

MODEL_NAME = next((m for m in PREFERRED if m in available), None)
if MODEL_NAME is None:
    # PREFERRED にないが使えるものを使う
    MODEL_NAME = next((m for m in available if 'flash' in m or 'pro' in m), None)
if MODEL_NAME is None:
    raise RuntimeError(f'利用可能なモデルが見つかりません。APIキーを確認してください。利用可能: {available}')

print(f'✅ APIキー確認済み  使用モデル: {MODEL_NAME}')

In [ ]:
# ── ステップ2：本のタイトルを入力 ──────────────────────────────────
BOOK_TITLE = input('本のタイトル: ').strip()
if not BOOK_TITLE:
    raise ValueError('タイトルを入力してください')
print(f'📖 対象: {BOOK_TITLE}')

In [ ]:
# ── ステップ3：AI呼び出し関数を定義 ────────────────────────────────
import json, re, time

SYSTEM_PROMPT = """あなたは「賢者とユイの対話形式」で本の本質を伝える人気ライターです。

## キャラクター設定

**賢者（けんじゃ）**
- 50代の穏やかな老哲学者
- 難しい概念をシンプルなたとえ話で説明する
- 口癖：「そうじゃな」「面白い視点じゃ」「核心を突いておるな」

**ユイ（ゆい）**
- 20代の好奇心旺盛な読者の代弁者
- 読者が「自分も同じこと思ってた！」と共感できる存在
- 口癖：「なるほど！」「えっ、それってどういうことですか？」「つまり〜ってことですね！」

## 執筆ルール
1. 対話形式（賢者とユイの会話）で書く
2. 各章は2000〜3000文字を目安に
3. 具体的な例やたとえ話を豊富に使う
4. 著作権に配慮し、本の文章を直接引用せず、テーマや考え方を自分の言葉で解説する
"""

_model = genai.GenerativeModel(
    model_name=MODEL_NAME,
    system_instruction=SYSTEM_PROMPT,
    generation_config=genai.GenerationConfig(temperature=0.9, max_output_tokens=4096),
)
_last_call = 0.0
CALL_INTERVAL = 4  # 秒

def call_gemini(prompt: str, max_tokens: int = 4096) -> str:
    global _last_call
    wait = CALL_INTERVAL - (time.time() - _last_call)
    if wait > 0:
        time.sleep(wait)
    for attempt, backoff in enumerate([0, 30, 60, 120]):
        if backoff:
            print(f'  ⚠️ レート制限 — {backoff}秒待機中...')
            time.sleep(backoff)
        try:
            _last_call = time.time()
            cfg = genai.GenerationConfig(temperature=0.9, max_output_tokens=max_tokens)
            res = _model.generate_content(prompt, generation_config=cfg)
            return res.text
        except Exception as e:
            if '429' in str(e) and attempt < 3:
                continue
            raise
    raise RuntimeError('Gemini API 呼び出しに失敗しました')

def extract_json(text: str) -> dict:
    m = re.search(r'```(?:json)?\s*([\s\S]+?)\s*```', text)
    if m:
        return json.loads(m.group(1))
    m = re.search(r'\{[\s\S]+\}', text)
    if m:
        return json.loads(m.group(0))
    raise ValueError('JSONが見つかりません')

print('✅ 関数定義完了')

In [ ]:
# ── ステップ4：書籍情報を自動取得 ──────────────────────────────────
print(f'🔍 書籍情報を検索中: {BOOK_TITLE} ...')

info_prompt = f"""以下の書籍タイトルの情報をJSON形式で回答してください。

タイトル: {BOOK_TITLE}

```json
{{
  "author": "著者名（不明なら不明）",
  "category": "ジャンル（ビジネス/自己啓発/投資/心理学/小説/歴史/科学 など）",
  "description": "本の内容を3〜5文で説明"
}}
```
実在する書籍なら正確に、不明な場合は推測で構いません。"""

book_info = extract_json(call_gemini(info_prompt, max_tokens=512))
BOOK_AUTHOR      = book_info.get('author', '不明')
BOOK_CATEGORY    = book_info.get('category', 'ビジネス')
BOOK_DESCRIPTION = book_info.get('description', '')

print('✅ 書籍情報を取得しました')
print(f'   著者    : {BOOK_AUTHOR}')
print(f'   カテゴリ: {BOOK_CATEGORY}')
print(f'   説明    : {BOOK_DESCRIPTION[:80]}...' if len(BOOK_DESCRIPTION) > 80 else f'   説明    : {BOOK_DESCRIPTION}')

In [ ]:
# ── ステップ5：章構成を計画 ─────────────────────────────────────────
print(f'📚 章構成を計画中...')

plan_prompt = f"""以下の本について賢者とユイの対話形式の解説本を作ります。

## 対象書籍
タイトル: {BOOK_TITLE} / 著者: {BOOK_AUTHOR}
カテゴリ: {BOOK_CATEGORY}
説明: {BOOK_DESCRIPTION or '（なし）'}

以下のJSON形式で出力してください：
```json
{{
  "book_title": "【賢者とユイが語る】〇〇の本質",
  "subtitle": "〇〇が教えてくれる人生の知恵",
  "description": "本の説明文（300文字程度、電子書籍ストア投稿用）",
  "keywords": ["キーワード1", "キーワード2", "キーワード3"],
  "chapter_titles": ["第1章タイトル", "第2章タイトル", "第3章タイトル",
                      "第4章タイトル", "第5章タイトル", "第6章タイトル"]
}}
```
章は5〜7章構成にしてください。"""

plan = extract_json(call_gemini(plan_prompt, max_tokens=1024))
print('✅ 構成完了')
print(f'   タイトル   : {plan["book_title"]}')
print(f'   サブタイトル: {plan["subtitle"]}')
for i, t in enumerate(plan['chapter_titles'], 1):
    print(f'   第{i}章: {t}')

In [ ]:
# ── ステップ6：まえがき執筆 ──────────────────────────────────────────
print('✍️  まえがきを執筆中...')

foreword = call_gemini(f"""『{BOOK_TITLE}』（著：{BOOK_AUTHOR}）の解説書のまえがきを書いてください。
- この本を手に取った読者へのメッセージ
- 賢者とユイというキャラクターの紹介
- この解説本で得られること
- 400〜600文字、マークダウン形式""")

print('✅ まえがき完了')

In [ ]:
# ── ステップ7：各章を執筆 ────────────────────────────────────────────
chapters = []
toc_str = '\n'.join(f'{i+1}. {t}' for i, t in enumerate(plan['chapter_titles']))

for i, title in enumerate(plan['chapter_titles'], 1):
    print(f'✍️  第{i}章「{title}」を執筆中...')
    prompt = f"""『{BOOK_TITLE}』（著：{BOOK_AUTHOR}）の解説本の第{i}章を書いてください。

章タイトル: {title}
全章構成:
{toc_str}

注意：
- 本の文章を直接引用しない（著作権配慮）
- テーマ・考え方を自分の言葉で解説する
- 賢者とユイの自然な対話形式
- 2000〜3000文字
- マークダウン形式（**ユイ**：〜 / **賢者**：〜）"""

    content = call_gemini(prompt, max_tokens=4096)
    chapters.append({'number': i, 'title': title, 'content': content})
    print(f'   ✅ 完了 ({len(content):,}文字)')

print(f'\n✅ 全{len(chapters)}章の執筆完了！')

In [ ]:
# ── ステップ8：あとがき執筆 ──────────────────────────────────────────
print('✍️  あとがきを執筆中...')

afterword = call_gemini(f"""『{BOOK_TITLE}』解説本のあとがきを書いてください。
全章を通じたメッセージの総括と読者へのエール。
400〜600文字、マークダウン形式。
全章タイトル:
{toc_str}""")

print('✅ あとがき完了')

In [ ]:
# ── ステップ9：Markdownファイル保存 ────────────────────────────────
import os
from pathlib import Path

safe_title = re.sub(r'[\\/*?:"<>|【】\s]', '_', plan['book_title'])[:50]
out = Path('/content/ebook_output')
out.mkdir(exist_ok=True)

parts = [
    f"# {plan['book_title']}\n",
    f"**{plan['subtitle']}**\n",
    '著者：賢者とユイの読書倶楽部\n',
    '---\n',
    '## まえがき\n', foreword, '\n---\n',
]
for ch in chapters:
    parts += [f"## 第{ch['number']}章　{ch['title']}\n", ch['content'], '\n---\n']
parts += ['## あとがき\n', afterword]

full_text = '\n'.join(parts)
md_path = out / f'{safe_title}.md'
md_path.write_text(full_text, encoding='utf-8')

total_chars = len(full_text)
print(f'✅ Markdown保存: {md_path}  ({total_chars:,}文字)')

In [ ]:
# ── ステップ10：EPUB生成 ────────────────────────────────────────────
import html as html_module
from ebooklib import epub

CSS = b'body{font-family:serif;line-height:1.9;margin:2em;}h1,h2,h3{margin-top:1.5em;}p{margin:.5em 0;}strong{font-weight:bold;}'

def md_to_html(text: str) -> str:
    out = []
    for line in text.split('\n'):
        line = html_module.escape(line)
        if line.startswith('### '):
            line = f'<h3>{line[4:]}</h3>'
        elif line.startswith('## '):
            line = f'<h2>{line[3:]}</h2>'
        elif line.startswith('# '):
            line = f'<h1>{line[2:]}</h1>'
        elif line.strip() in ('', '---'):
            line = '<br/>'
        else:
            line = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', line)
            line = f'<p>{line}</p>'
        out.append(line)
    return '\n'.join(out)

bk = epub.EpubBook()
bk.set_identifier('id_' + safe_title)
bk.set_title(plan['book_title'])
bk.set_language('ja')
bk.add_author('賢者とユイの読書倶楽部')

css_item = epub.EpubItem(uid='style', file_name='style/main.css', media_type='text/css', content=CSS)
bk.add_item(css_item)

spine, toc = ['nav'], []

def make_chapter(uid, fname, title, body_html):
    c = epub.EpubHtml(title=title, file_name=fname, lang='ja')
    c.content = f'<html><body><h2>{html_module.escape(title)}</h2>{body_html}</body></html>'
    c.add_item(css_item)
    bk.add_item(c)
    spine.append(c)
    toc.append(epub.Link(fname, title, uid))

make_chapter('foreword', 'foreword.xhtml', 'まえがき', md_to_html(foreword))
for ch in chapters:
    make_chapter(f'ch{ch["number"]}', f'ch{ch["number"]}.xhtml',
                 f'第{ch["number"]}章　{ch["title"]}', md_to_html(ch['content']))
make_chapter('afterword', 'afterword.xhtml', 'あとがき', md_to_html(afterword))

bk.toc = toc
bk.spine = spine
bk.add_item(epub.EpubNcx())
bk.add_item(epub.EpubNav())

epub_path = out / f'{safe_title}.epub'
epub.write_epub(str(epub_path), bk)
print(f'✅ EPUB保存: {epub_path}  ({epub_path.stat().st_size/1024:.1f} KB)')

In [ ]:
# ── ステップ11：Word(.docx)生成 ─────────────────────────────────────
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH

doc = Document()

# タイトルページ
h = doc.add_heading(plan['book_title'], 0)
h.alignment = WD_ALIGN_PARAGRAPH.CENTER
p = doc.add_paragraph(plan['subtitle'])
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
doc.add_paragraph('著者：賢者とユイの読書倶楽部').alignment = WD_ALIGN_PARAGRAPH.CENTER
doc.add_page_break()

def add_md(doc, text):
    for line in text.split('\n'):
        if line.startswith('## '):
            doc.add_heading(line[3:], level=2)
        elif line.startswith('# '):
            doc.add_heading(line[2:], level=1)
        elif line.strip() in ('', '---'):
            doc.add_paragraph('')
        else:
            para = doc.add_paragraph()
            for part in re.split(r'(\*\*.+?\*\*)', line):
                if part.startswith('**') and part.endswith('**'):
                    para.add_run(part[2:-2]).bold = True
                else:
                    para.add_run(part)

doc.add_heading('まえがき', level=1)
add_md(doc, foreword)
doc.add_page_break()

for ch in chapters:
    doc.add_heading(f'第{ch["number"]}章　{ch["title"]}', level=1)
    add_md(doc, ch['content'])
    doc.add_page_break()

doc.add_heading('あとがき', level=1)
add_md(doc, afterword)

docx_path = out / f'{safe_title}.docx'
doc.save(str(docx_path))
print(f'✅ Word保存: {docx_path}  ({docx_path.stat().st_size/1024:.1f} KB)')

In [ ]:
# ── ステップ12：ダウンロード ────────────────────────────────────────
from google.colab import files

print('=' * 50)
print('🎉 完成！')
print('=' * 50)
print(f'📖 タイトル : {plan["book_title"]}')
print(f'📝 総文字数 : {total_chars:,} 文字')
print(f'📚 章数     : {len(chapters)} 章')
print()
print('ダウンロードを開始します...')
files.download(str(md_path))
files.download(str(epub_path))
files.download(str(docx_path))
print('✅ 3ファイルのダウンロード完了')
print('   .md   — テキスト原稿')
print('   .epub — 電子書籍ストア投稿用')
print('   .docx — Googleドキュメントで編集可')